In [ ]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
import random
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using {device}")
import pandas as pd 
df = pd.read_csv('IMDB Dataset.csv')
print(df.head())
print(f"shape: {df.shape}")
print(df['sentiment'].value_counts())
df["label"] = (df['sentiment'] == 'positive').astype(int)
texts = df['review'].tolist()
labels = df['label'].tolist()
def clean_text(text):
    text = re.sub(r'<.?>', '', text)
    text = text .lower()
    text = re.sub(r'[^a-z\s]','', text)
    return text.strip()
texts = [clean_text(t) for t in texts]
print(f"sample : {texts[0][:100]}")

def tokenize(text):
    return text.split()



def build_vocab(texts, max_vocab=25000, min_freq = 2):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word , count in counter.most_common(max_vocab - 2):
        if count < min_freq:
            break
        vocab[word] = len(vocab)
    return vocab
vocab = build_vocab(texts)

print(f"Vocab size: {len(vocab):,}")

def encode(text,vocab, max_len = 300):
    tokens = tokenize(text)
    ids = [vocab.get(t,1) for t in tokens]
    ids = ids[:max_len]
    ids = ids + [0] * (max_len - len(ids))
    return ids

class IMDBDatasets(Dataset):
    def __init__(self, texts, labels , vocab , max_len=300):
        self.data = [encode(t, vocab, max_len) for t in texts]
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return (
            torch.tensor(self.data[idx],dtype= torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float)
        )
split = int(0.8 *  len(texts))
train_dataset = IMDBDatasets(texts[:split], labels[:split], vocab)
test_dataset  = IMDBDatasets(texts[split:], labels[split:], vocab)
train_loader = DataLoader(train_dataset, batch_size= 64 , shuffle= True)
test_loader = DataLoader(train_dataset, batch_size= 64, shuffle= False)
print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)}")
class ImprovedSentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size,num_layers):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings= vocab_size,
            embedding_dim = embed_dim,
            padding_idx = 0
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size= hidden_size,
            num_layers= num_layers,
            batch_first= True,
            dropout= 0.5,
            bidirectional=True


        )

        self.fc = nn.Linear(hidden_size *  2, 1)
        self.dropout = nn.Dropout(0.5)
        self.sigmoid = nn.Sigmoid()
    def forward(self,x):
        embedded = self.dropout(self.embedding(x))
        out, _ = self.lstm(embedded)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        out = self.sigmoid(out)
        return out.squeeze(1)
model = ImprovedSentimentLSTM(
    vocab_size  = len(vocab),
    embed_dim   = 128,
    hidden_size = 256,
    num_layers  = 2
).to(device)
criterion = nn.BCELoss() 
optimizer = optim.Adam(model.parameters(), lr = 0.001)
print(f"parameters : {sum(p.numel() for p in model.parameters())}")
epochs = 2
best_acc = 0
print("\n " + "=" * 60)
print(f"{'Epoch':<8}{'Train Loss':>12}{'Train Acc':>12}{'Test Acc':>12}")
print("="*60)
for epoch in range(epochs):
    model.train()
    total_loss = correct = total = 0
    for texts_b, labels_b in train_loader:
        texts_b, labels_b = texts_b.to(device), labels_b.to(device)
        outputs = model(texts_b)
        loss    = criterion(outputs, labels_b)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        predicted = (outputs > 0.5).float()
        correct +=(predicted == labels_b).sum().item()
        total += labels_b.size(0)
    train_loss = total_loss/ len(train_loader)
    train_acc = correct / total * 100
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for texts_b, labels_b in test_loader:
            texts_b, labels_b = texts_b.to(device), labels_b.to(device)
            outputs   = model(texts_b)
            predicted = (outputs > 0.5).float()
            correct  += (predicted == labels_b).sum().item()
            total    += labels_b.size(0)

    test_acc = correct / total * 100

    # save best model!
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'best_sentiment.pth')
        star = "⭐"
    else:
        star = ""

    print(f"{epoch+1:<8}{train_loss:>12.4f}{train_acc:>11.2f}%"
          f"{test_acc:>11.2f}% {star}")

print("="*60)
print(f"\nBest Test Accuracy: {best_acc:.2f}%")

# ---- PREDICT ----
model.load_state_dict(torch.load('best_sentiment.pth'))
model.eval()

def predict(review):
    encoded  = torch.tensor(
        [encode(clean_text(review), vocab)],
        dtype=torch.long
    ).to(device)
    with torch.no_grad():
        prob = model(encoded).item()
    label = "POSITIVE 😊" if prob > 0.5 else "NEGATIVE 😡"
    print(f"{label} ({prob:.3f}) | {review[:60]}")
    print("\nPredictions:")
print("-"*60)
predict("This movie was absolutely incredible! Best film ever!")
predict("Terrible waste of time. Boring and poorly made.")
predict("Great acting but the story was a bit slow.")
predict("One of the worst movies I have ever seen in my life.")
predict("Masterpiece! Every plot was iroic  and deep.")

